In [0]:
print("Spark Version:", spark.version)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define the schema [cite: 322-327]
schema = StructType([
    StructField("flight_id", StringType(), True),
    StructField("aircraft_type", StringType(), True),
    StructField("delay_minutes", IntegerType(), True),
    StructField("status", StringType(), True)
])

# Dummy data with a duplicate and a NULL [cite: 544, 404]
data = [
    ("FL001", "Boeing 737", 75, "COMPLETED"),
    ("FL002", "Airbus A320", 15, "COMPLETED"),
    ("FL003", "Boeing 787", None, "CANCELLED"),
    ("FL001", "Boeing 737", 75, "COMPLETED"), # Duplicate [cite: 544]
    ("FL004", "Airbus A350", 45, "COMPLETED")
]

# Create DataFrame
df = spark.createDataFrame(data, schema)
df.show()

In [0]:
# 1. Drop duplicates [cite: 544]
# 2. Fill NULL delays with 0 [cite: 413, 547-548]
silver_df = df.dropDuplicates().fillna(0, subset=["delay_minutes"])

silver_df.show()

In [0]:
# Create a Risk Tier based on delay minutes [cite: 433-436, 549-552]
triage_df = silver_df.withColumn("risk_tier", 
    F.when(F.col("delay_minutes") > 60, "HIGH")
    .when(F.col("delay_minutes") > 20, "MEDIUM")
    .otherwise("LOW")
)

triage_df.show()

In [0]:
# Group by aircraft type and find average delay [cite: 558-561, 356]
gold_df = triage_df.groupBy("aircraft_type").agg(
    F.count("flight_id").alias("total_flights"), # [cite: 354, 559]
    F.avg("delay_minutes").alias("avg_delay")   # [cite: 356, 561]
)

gold_df.show()

In [0]:
from pyspark.sql import Window

# Define the window [cite: 369-370]
windowSpec = Window.partitionBy("aircraft_type").orderBy(F.desc("delay_minutes"))

# Apply Rank [cite: 374]
ranked_df = triage_df.withColumn("rank", F.rank().over(windowSpec))

ranked_df.filter(F.col("rank") == 1).show()

NEW DATA. 

In [0]:
data = [
("E001","A320-101","A320","BOM","DEL","2024-01-01",15,"CLEAR","AVAILABLE",0,180),
("E002","A320-101","A320","DEL","BLR","2024-01-02",65,"FOG","AVAILABLE",1,165),
("E003","B737-202","B737","BLR","HYD","2024-01-01",5,"CLEAR","AVAILABLE",0,150),
("E004","B737-202","B737","HYD","DEL","2024-01-02",80,"RAIN","SHORTAGE",1,140),
("E005","A320-101","A320","DEL","BOM","2024-01-03",120,"STORM","AVAILABLE",1,170),
("E006","A321-303","A321","BOM","MAA","2024-01-01",0,"CLEAR","AVAILABLE",0,200),
("E007","A321-303","A321","MAA","BOM","2024-01-02",25,"CLEAR","AVAILABLE",0,195),
("E008","B737-202","B737","DEL","BOM","2024-01-03",45,"FOG","AVAILABLE",0,160),
("E009","A320-101","A320","BOM","DEL","2024-01-04",90,"STORM","SHORTAGE",1,175),
("E010","A321-303","A321","BOM","DEL","2024-01-03",10,"CLEAR","AVAILABLE",0,210)
]

columns = [
"event_id","aircraft_id","aircraft_type","origin_airport","destination_airport",
"flight_date","delay_minutes","weather_status","crew_status","maintenance_flag","passengers"
]

df = spark.createDataFrame(data, columns)

In [0]:
(df.show(5))

In [0]:
#Select & Rename: Select the event_id, aircraft_id, and passengers. Rename the passengers column to total_passengers.
df1 = df.select("event_id","aircraft_id","passengers").withColumnRenamed("passengers","Total_passengers")
df1.show(3)

In [0]:
#Unique Values: Find all unique aircraft_type values present in the dataset.
df.select("aircraft_type").distinct().show()
df.groupBy("aircraft_type").count().show()

In [0]:
#Basic Sorting: Sort the dataframe by flight_date in ascending order and then by delay_minutes in descending order.
df.orderBy("flight_date",df["delay_minutes"].desc()).show()

In [0]:
#Weather Alert: Filter for all flights where the weather_status is either "STORM" or "FOG".
df.filter(df.weather_status.isin(["STORM","FOG"])).show()

In [0]:
#Critical Maintenance: Find all flights where maintenance_flag is 1 AND the delay_minutes are greater than 60.
df.filter((df.maintenance_flag==1) & (df.delay_minutes>60)).show()

In [0]:
#Crew Shortage: Select only the event_id and origin_airport for flights where the crew_status is "SHORTAGE".
df.select("event_id","origin_airport").filter(df.crew_status=="SHORTAGE").show()

In [0]:
#New Column (Calculated): Create a new column called is_delayed that returns True if delay_minutes > 15, and False otherwise.
df.withColumn("is_delayed",df.delay_minutes>15).show()

In [0]:
#String Manipulation: Select the aircraft_id and create a new column called fleet_code that only contains the first 4 characters of the aircraft_id (e.g., "A320").
df.withColumn("fleet_code",df.aircraft_id.substr(0,4)).show()

In [0]:
#Retrieve the top 3 most delayed flights starting from "BOM" (Mumbai).
df.filter(df.origin_airport=="BOM").orderBy(df.delay_minutes.desc()).limit(3).show()

In [0]:
#Null Check Simulation: Even though this data is clean, write a query to filter out any rows where flight_date might be null (Standard practice in a Silver-to-Gold pipeline).
df.filter(df.flight_date.isNull()).show()

In [0]:
airport_data = [
    ("BOM", "Chhatrapati Shivaji Maharaj International", "Mumbai", "Tier 1"),
    ("DEL", "Indira Gandhi International", "Delhi", "Tier 1"),
    ("BLR", "Kempegowda International", "Bangalore", "Tier 1"),
    ("HYD", "Rajiv Gandhi International", "Hyderabad", "Tier 1"),
    ("MAA", "Chennai International", "Chennai", "Tier 1")
]

airport_columns = ["airport_code", "airport_name", "city", "airport_category"]
airports_df = spark.createDataFrame(airport_data, airport_columns)

In [0]:
airports_df.show()